In [2]:
!pip3 install --upgrade openai
import openai
from openai import OpenAI
!pip3 install pandas

In [ ]:

import pandas as pd

openai.api_key = ""
openai.api_base = "https://api.cohere.ai/compatibility/v1" 

train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('validation.csv')
test_df = pd.read_csv('test.csv')

print(f'Training set: {train_df.shape[0]} samples')
print(f'Validation set: {val_df.shape[0]} samples')
print(f'Test set: {test_df.shape[0]} samples')
print(train_df.head(), "\n\n")

print("Training dataset columns:", train_df.columns.tolist())
print("Validation dataset columns:", val_df.columns.tolist())

print("\nFirst few rows of the training dataset:")
print(train_df.head())

Training set: 56355 samples
Validation set: 8421 samples
Test set: 15878 samples
   phase                                           question  \
0      1    Tell me what the notes are for South Australia    
1      1  What is the current series where the new serie...   
2      1            What is the format for South Australia?   
3      1  Name the background colour for the Australian ...   
4      1      how many times is the fuel propulsion is cng?   

                                               table  \
0  {'header': array(['State/territory', 'Text/bac...   
1  {'header': array(['State/territory', 'Text/bac...   
2  {'header': array(['State/territory', 'Text/bac...   
3  {'header': array(['State/territory', 'Text/bac...   
4  {'header': array(['Order Year', 'Manufacturer'...   

                                                 sql  
0  {'human_readable': 'SELECT Notes FROM table WH...  
1  {'human_readable': 'SELECT Current series FROM...  
2  {'human_readable': 'SELECT Format F

In [ ]:
import re

def get_table_info(df, row):

    header_list = df['table'][row].split('array(')[1]
    cleaned_str = re.sub(r'\].*', ']', header_list)
    header_str = cleaned_str.replace('\n       ', ' ')
    cleaned_str = 'Header: ' + header_str[:header_str.find(']') + 1]

    rows_pattern = r"array\((.*?)\], dtype=object\)"
    matches = re.findall(rows_pattern, df['table'][4], re.DOTALL)

    rows_2d = []
    for match in matches:
        if 'header' in match or 'page_title' in match:
            continue
        row = match.strip().replace("dtype=object", "").replace("array([", "").replace("],", ",")
        row_list = [item.strip().strip("'") for item in row.split(",")]
        rows_2d.append(row_list[2:])
    rows_2d = 'Rows: ' + str(rows_2d)
    return cleaned_str, rows_2d


def create_few_shot_prompt(user_question, question_dict, examples, question_idx, num_examples=3, question_col='question', query_col='sql'):
    prompt = "You are a SQL generator. Given the following examples, generate the corresponding SQL query for the provided natural language question. Respond with only one query. Make sure your response ends in a character and not in symbols such as quotes or semi-colons. \n"
    prompt += "Make sure you DO NOT INCLUDE THE KEYWORD 'DISTINCT' as part of your query. \
    Also make sure you DO NOT USE COUNT(*) and instead use count of the column name. \
    DO NOT USE 'like'. Instead use '='. \
    Double check that the table and variable names match up with the associated table provided. \
    Lastly, DO NOT START THE QUERY WITH 'sql'. ENSURE THAT THERE ARE NO TRAILING LINES OR SPACES IN FRONT OR AFTER THE QUERY; ONLY return the raw query beginning with 'SELECT' \n\n"
    
    prompt += "You are also given an associated table with the following columns and data, which you must use in your query.\n"
    
    for i in range(min(num_examples, len(examples))):

        row = examples.iloc[i]
        prompt += f"Example {i+1}:\n"
        prompt += f"Question: {row[question_col]}\n"
        cols, vals = get_table_info(examples, i)
        prompt += cols
        prompt += '\n'
        prompt += vals
        prompt += f"SQL: {row[query_col]}\n\n"
    
    prompt += "Now, generate the SQL query for the following question:\n"
    prompt += f"Question: {user_question}\nSQL:"
    cols, vals = get_table_info(question_dict, question_idx)
    prompt += cols
    prompt += '\n'
    prompt += vals
    
    return prompt


In [122]:
import re

def get_sql_from_cohere(prompt, temperature=0.0, max_tokens=150):
    client = OpenAI(api_key=openai.api_key, base_url="https://api.cohere.ai/compatibility/v1")
    response = client.chat.completions.create(
        model="command-a-03-2025",
        messages=[
            {"role": "system", "content": "You are a helpful assistant specialized in generating SQL queries."},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    sql_query = response.choices[0].message.content
    return sql_query

In [123]:
import re

if 'question' in val_df.columns:
    idx = 0
    sample_question = val_df.iloc[idx]['question']
    table = get_table_info(val_df, idx)
else:
    sample_question = val_df.iloc[0]['text']

idx = 0

print("Sample Question from validation dataset:")
print(sample_question)

prompt = create_few_shot_prompt(sample_question, val_df, train_df, question_idx=idx, num_examples=3, question_col='question', query_col='sql')

print("\nConstructed Prompt:\n")
print(prompt)

generated_sql = get_sql_from_cohere(prompt)

print("\nGenerated SQL Query:\n")
print(generated_sql)

Sample Question from validation dataset:
What position does the player who played for butler cc (ks) play?

Constructed Prompt:

You are a SQL generator. Given the following examples, generate the corresponding SQL query for the provided natural language question. Respond with only one query. Make sure your response ends in a character and not in symbols such as quotes or semi-colons. 
Make sure you DO NOT INCLUDE THE KEYWORD 'DISTINCT' as part of your query. Also make sure you DO NOT USE COUNT(*) and instead use count of the column name. 

You are also given an associated table with the following columns and data, which you must use in your query.
Example 1:
Question: Tell me what the notes are for South Australia 
Header: ['State/territory', 'Text/background colour', 'Format', 'Current slogan', 'Current series', 'Notes']
Rows: [['Phantom (High Floor)', '444-464 (21)', 'DD S50EGR Allison WB-400R', 'Diesel'], ['Phantom (High Floor)', '465-467 (3)', 'DD S50 Allison WB-400R', 'Diesel'], 

In [141]:
import time

def levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    
    previous_row = list(range(len(s2) + 1))
    
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

def normalize_sql(query):
    if query is None:
        return ""

    query = " ".join(query.strip().lower().split())

    for ch in ["'", '"', "`", "[", "]", "\\", "(", ")", ","]:
        query = query.replace(ch, "")
    
    if "where" not in query:
        return query.replace(" ", "")

    before_where, after_where = query.split("where", 1)

    rest_clause = ""
    for clause in ["group by", "order by", "limit"]:
        if clause in after_where:
            where_conditions, rest = after_where.split(clause, 1)
            rest_clause = clause + " " + rest.strip()
            break
    else:
        where_conditions = after_where

    conditions = [cond.strip().replace(" ", "") for cond in where_conditions.split("and")]
    conditions.sort()
    sorted_where = "and".join(conditions)

    cleaned_query = before_where.replace(" ", "") + "where" + sorted_where
    if rest_clause:
        cleaned_query += rest_clause.replace(" ", "")
    
    return cleaned_query

def evaluate_sql_strict(generated_query, expected_query):
    return generated_query == expected_query

def validate_accuracy(expected, generated, threshold=5):
    distance = levenshtein_distance(expected, generated)
    return distance <= threshold, distance

In [146]:
total_queries = len(val_df)
correct_queries = 0
results = []

sample_size = 20
for index, row in val_df.head(sample_size).iterrows():
    if index != 0 and index % 10 == 0:
        #need to do this since free cohere api can only do 10 calls/min
        time.sleep(61)
    prompt = row['question']
    prompt = create_few_shot_prompt(prompt, val_df, train_df, question_idx=index, num_examples=3, question_col='question', query_col='sql')
    expected_query = row['sql']
    match = re.search(r"'human_readable': '(.*?)'", expected_query)
    if match == None:
        match = re.search(r"'human_readable': *\"(.*?)\"", expected_query)
    if match:
        expected_query = match.group(1)
    else:
        expected_query = None
                                   
    expected_query = normalize_sql(expected_query)
    print("\n", "EXPECTED: ", expected_query)

    generated_query = get_sql_from_cohere(prompt)
    generated_query = normalize_sql(generated_query)

    print("GENERATED: ", generated_query)

    is_correct = evaluate_sql_strict(generated_query, expected_query)
    results.append({
        "prompt": prompt,
        "generated_query": generated_query,
        "expected_query": expected_query,
        "correct": is_correct
    })

    is_acceptable, diff = validate_accuracy(expected_query, generated_query)
    if is_correct:
        correct_queries += 1
    elif is_acceptable:
        correct_queries += 1
    else:

        print("INCORRECT!!!!!! \n")

accuracy = (correct_queries / sample_size) * 100
print(f"Accuracy: {accuracy:.2f}% ({correct_queries}/{sample_size} correct)")


 EXPECTED:  selectpositionfromtablewhereschool/clubteam=butlerccks
GENERATED:  selectpositionfromtablewhereschool/clubteam=butlerccks

 EXPECTED:  selectcountschool/clubteamfromtablewhereno.=3
GENERATED:  selectcountschool/clubteamfromtablewhereno.=3

 EXPECTED:  selectschool/clubteamfromtablewhereno.=21
GENERATED:  selectschool/clubteamfromtablewhereno.=21

 EXPECTED:  selectplayerfromtablewhereno.=42
GENERATED:  selectplayerfromtablewhereno.=42

 EXPECTED:  selectplayerfromtablewhereposition=guardandyearsintoronto=1996-97
GENERATED:  selectplayerfromtablewhereposition=guardandyearsintoronto=1996-97

 EXPECTED:  selectplayerfromtablewhereschool/clubteam=westchesterhighschool
GENERATED:  selectplayerfromtablewhereschool/clubteam=westchesterhighschool

 EXPECTED:  selectschool/clubteamfromtablewhereplayer=amirjohnson
GENERATED:  selectschool/clubteamfromtablewhereplayer=amirjohnson

 EXPECTED:  selectcountno.fromtablewhereyearsintoronto=2005-06
GENERATED:  selectsumnofromtablewhereyear

In [144]:
sampled_df = test_df.sample(n=50, replace=False, random_state=42).reset_index(drop=True)

total_queries = len(sampled_df)
correct_queries = 0
results = []

sample_size = 50
for index, row in sampled_df.iterrows():
    if index != 0 and index % 10 == 0:
        #need to do this since free cohere api can only do 10 calls/min
        time.sleep(61)
    prompt = row['question']
    prompt = create_few_shot_prompt(prompt, sampled_df, train_df, question_idx=index, num_examples=3, question_col='question', query_col='sql')
    expected_query = row['sql']
    match = re.search(r"'human_readable': '(.*?)'", expected_query)
    if match == None:
        match = re.search(r"'human_readable': *\"(.*?)\"", expected_query)
    if match:
        expected_query = match.group(1)
    else:
        expected_query = None
                                   
    expected_query = normalize_sql(expected_query)
    print("\n", "EXPECTED: ", expected_query)

    generated_query = get_sql_from_cohere(prompt)
    generated_query = normalize_sql(generated_query)

    print("GENERATED: ", generated_query)

    is_correct = evaluate_sql_strict(generated_query, expected_query)
    results.append({
        "prompt": prompt,
        "generated_query": generated_query,
        "expected_query": expected_query,
        "correct": is_correct
    })

    is_acceptable, diff = validate_accuracy(expected_query, generated_query)
    if is_correct:
        correct_queries += 1
    elif is_acceptable:
        correct_queries += 1
    else:
        print("INCORRECT!!!!!! \n")

accuracy = (correct_queries / sample_size) * 100
print(f"Accuracy: {accuracy:.2f}% ({correct_queries}/{sample_size} correct)")


 EXPECTED:  selectsumdrawnfromtablewherelost=4
GENERATED:  selectdrawnfromtablewherelost=4

 EXPECTED:  selectratingfromtablewhererating/share18–49=0.9/4
GENERATED:  selectratingfromtablewhererating/share18–49=0.9/4

 EXPECTED:  selectmaxfirstelectedfromtable
GENERATED:  selectmaxfirstelectedfromtable

 EXPECTED:  selectcountpolesfromtablewherepoints=72
GENERATED:  selectcountpolesfromtablewherepoints=72

 EXPECTED:  selectmoto2winnerfromtablewheregrandprixandprix=shelladvancemalaysiangr
GENERATED:  selectmoto2winnerfromtablewheregrandprixandprix=shelladvancemalaysiangr

 EXPECTED:  selectcountincumbentfromtablewheredistrict=georgia8andparty=democratic
GENERATED:  selectcountincumbentfromtablewheredistrict=georgia8andparty=democratic

 EXPECTED:  selectmaxareamsrfromtablewhereareasq.deg.<291.045andfamily=perandrank>78
GENERATED:  selectmaxareamsrfromtablewhereareamsr<291.045andfamily=perandrank>78
INCORRECT!!!!!! 


 EXPECTED:  selectwomensdoublesfromtablewheremenssingles=kennethvella